# Chapter 2: LLM API Basics

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## 2.1 OpenAI API

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is an AI agent? Answer in one sentence."}
    ]
)

print(response.choices[0].message.content)

## 2.2 Anthropic API

In [ ]:
import anthropic

client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a helpful assistant.",
    messages=[
        {"role": "user", "content": "What is an AI agent? Answer in one sentence."}
    ]
)

print(message.content[0].text)

## 2.3 LiteLLM - Unified API

In [ ]:
from litellm import completion

# Call OpenAI via LiteLLM
response_openai = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is an AI agent? One sentence."}]
)
print("OpenAI:", response_openai.choices[0].message.content)

# Call Anthropic via LiteLLM
response_anthropic = completion(
    model="anthropic/claude-sonnet-4-20250514",
    messages=[{"role": "user", "content": "What is an AI agent? One sentence."}]
)
print("Anthropic:", response_anthropic.choices[0].message.content)

## 2.4 Conversation History

In [ ]:
from litellm import completion

# Multi-turn conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "My name is Alice."},
]

response = completion(model="gpt-4o-mini", messages=messages)
assistant_msg = response.choices[0].message.content
print("Assistant:", assistant_msg)

# Add assistant response and follow-up
messages.append({"role": "assistant", "content": assistant_msg})
messages.append({"role": "user", "content": "What is my name?"})

response2 = completion(model="gpt-4o-mini", messages=messages)
print("Assistant:", response2.choices[0].message.content)

## 2.5 Structured Output

In [ ]:
from litellm import completion
from pydantic import BaseModel

class MovieReview(BaseModel):
    title: str
    rating: float
    summary: str
    recommended: bool

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Review the movie 'Inception' by Christopher Nolan."}],
    response_format=MovieReview,
)

review = MovieReview.model_validate_json(response.choices[0].message.content)
print(f"Title: {review.title}")
print(f"Rating: {review.rating}")
print(f"Summary: {review.summary}")
print(f"Recommended: {review.recommended}")

## 2.6 Async Calls

In [ ]:
import asyncio
from litellm import acompletion

async def ask_model(model: str, question: str) -> str:
    response = await acompletion(
        model=model,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

async def main():
    question = "What is the capital of France? Answer in one word."
    models = ["gpt-4o-mini", "anthropic/claude-sonnet-4-20250514"]

    # Run calls concurrently
    results = await asyncio.gather(
        *[ask_model(m, question) for m in models]
    )

    for model, result in zip(models, results):
        print(f"{model}: {result}")

await main()

## 2.7 GAIA Benchmark Evaluation

In [ ]:
from datasets import load_dataset
from scratch_agent.eval.gaia import run_experiment, is_correct

# Load GAIA benchmark dataset
ds = load_dataset("gaia-benchmark/GAIA", "2023_all", split="validation")
problems = [row for row in ds]
print(f"Loaded {len(problems)} problems")

# Evaluate on a subset with multiple models
subset = problems[:5]
models = ["gpt-4o-mini", "anthropic/claude-sonnet-4-20250514"]

results = await run_experiment(subset, models)

# Print accuracy per model
for model, model_results in results.items():
    correct = sum(1 for r in model_results if r["correct"])
    total = len(model_results)
    print(f"{model}: {correct}/{total} ({correct/total*100:.1f}%)")